# 3. Turn recordings into batches

A batch is the group of recordings used in one forward/backward pass. The collator reads those recordings and prepares their audio features and text labels.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())

if "train_df" not in globals():
    run_notebook("01_data.ipynb")
if "model" not in globals():
    run_notebook("02_model.ipynb")

## One function prepares a batch

Labels are token IDs for the correct transcript. `-100` marks padding that should not affect the loss. Whisper adds the start token itself, so we remove it from the labels once.

In [ ]:
def prepare_batch(rows):
    audios = [read_audio(DATA_DIR / row["audio_path"]) for row in rows]
    texts = [row["text"] for row in rows]

    batch = processor.feature_extractor(
        audios, sampling_rate=SAMPLE_RATE,
        return_tensors="pt", return_attention_mask=True,
    )
    tokens = processor.tokenizer(texts, padding=True, return_tensors="pt")
    labels = tokens["input_ids"].masked_fill(tokens["attention_mask"] == 0, -100)
    if (labels[:, 0] == model.config.decoder_start_token_id).all():
        labels = labels[:, 1:]
    if labels.shape[1] > model.config.max_target_positions:
        raise ValueError("Transcript is too long; segment its audio and text together first.")
    batch["labels"] = labels
    return batch

## Create training and validation loaders

A loader gets rows from the table and calls `prepare_batch`. Training shuffles the rows; validation keeps their order. We compute features one batch at a time.

In [ ]:
train_loader = DataLoader(
    train_df.to_dict("records"), batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=prepare_batch, num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(
    validation_df.to_dict("records"), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=prepare_batch, num_workers=0,
)

## Try one forward pass

This checks that the tensors fit the model. It does not update weights. The loss measures disagreement with the correct transcript; lower is better.

In [ ]:
example_batch = prepare_batch(train_df.head(BATCH_SIZE).to_dict("records"))
print("Audio features:", tuple(example_batch["input_features"].shape))
print("Text labels:", tuple(example_batch["labels"].shape))

model.eval()
with torch.no_grad():
    outputs = model(**{name: tensor.to(DEVICE) for name, tensor in example_batch.items()})
example_loss = outputs.loss.item()
assert np.isfinite(example_loss), "The forward loss is invalid."
print("Example loss:", example_loss)
del outputs